In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import scipy.stats as stats

# Load dataset
file_path = "processed_dataset.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

# Data cleaning
df.drop_duplicates(inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)
df.fillna(df.mode().iloc[0], inplace=True)

# Data preparation
columns_to_exclude = ["Conductivity", "elongation", "Grade", "Grade_encoded"]
X = df.drop(columns=columns_to_exclude, errors="ignore")
y = df["UTS"]

# Feature scaling
scaler = MinMaxScaler()
X[X.select_dtypes(include=["float64", "int64"]).columns] = scaler.fit_transform(X.select_dtypes(include=["float64", "int64"]))

# Encode categorical variables
for col in X.select_dtypes(include="object").columns:
    X[col] = X[col].astype("category")

# Split dataset
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Hyperparameters
params_uts = {
    "n_estimators": 400,
    "learning_rate": 0.015,
    "max_depth": 2,
    "num_leaves": 8,
    "min_child_samples": 450,
    "colsample_bytree": 0.6,
    "feature_fraction": 0.6,
    "bagging_fraction": 0.5,
    "bagging_freq": 5,
    "lambda_l1": 10.0,
    "lambda_l2": 10.0,
    "random_state": 42
}

# Train model
lgb_model_uts = lgb.LGBMRegressor(**params_uts)
lgb_model_uts.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], eval_metric="rmse", callbacks=[lgb.early_stopping(50)])

# Predictions
y_pred = lgb_model_uts.predict(X_test)

c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=0.6 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] lambda_l1 is set=10.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=10.0
[LightGBM] [Warning] lambda_l2 is set=10.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=10.0
[LightGBM] [Warning] bagging_fraction is set=0.5, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=0.6 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] lambda_l1 is set=10.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=10.0
[LightGBM] [Warning] lambda_l2 is set=10.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=10.0
[LightGBM] [Warning] bagging_fraction is set=0.5, subsample=1.0 will be ignored. Current value: bagging_f

In [2]:
def predict_uts(input_data):
    import pandas as pd
    import numpy as np

    # Convert to DataFrame
    input_df = pd.DataFrame([input_data])

    # Ensure input has all features used during training
    missing_cols = set(X.columns) - set(input_df.columns)
    for col in missing_cols:
        input_df[col] = 0  # Fill missing columns with 0 or an appropriate default

    # Reorder columns to match training data exactly
    input_df = input_df[X.columns]

    # Apply scaling
    numeric_cols = X.select_dtypes(include=["float64", "int64"]).columns
    input_df[numeric_cols] = scaler.transform(input_df[numeric_cols].values)

    # Predict UTS
    y_pred_input = lgb_model_uts.predict(input_df)

    # Print prediction
    print(f"Predicted UTS: {y_pred_input[0]:.4f}")

# Example new input
new_input = {
    "%FE": 0.18,
    "emulsion_temp": 62.3,
    "emulsion_pr": 1.85,
    "rod_quench_cw_exit": 6.8,
    "casting_wheel_rpm": 2.0,
    "rod_quench_cw_entry": 412.0,
    "rm_motor_cooling_water_pressure": 1.7,
    "rolling_mill_amp": 305.0,
    "cool_water_flow": 125.5,
    "cooling_water_pressure": 4.5,
    "cooling_water_temp": 37.0,
    "rolling_mill_rpm": 770.0
}

# Predict for new input
predict_uts(new_input)


[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=0.6 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] lambda_l1 is set=10.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=10.0
[LightGBM] [Warning] lambda_l2 is set=10.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=10.0
[LightGBM] [Warning] bagging_fraction is set=0.5, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
Predicted UTS: 9.0043


c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [3]:
import joblib
# Save the trained LightGBM model
model_filename = "lgb_model_uts.pkl"
joblib.dump(lgb_model_uts, model_filename)
print(f"Model saved as {model_filename}")
import joblib
# Save the trained scaler
joblib.dump(scaler, "scaler.pkl")
print("Scaler saved successfully as scaler.pkl")
import joblib
joblib.dump(X, "features.pkl")

Model saved as lgb_model_uts.pkl
Scaler saved successfully as scaler.pkl


['features.pkl']

In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import scipy.stats as stats

# Load dataset
file_path = "processed_dataset.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

# Data cleaning
df.drop_duplicates(inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)
df.fillna(df.mode().iloc[0], inplace=True)

# Data preparation
columns_to_exclude = ["UTS", "Conductivity", "Grade", "Grade_encoded"]
X = df.drop(columns=columns_to_exclude, errors="ignore")
y = df["elongation"]

# Feature scaling
scaler = MinMaxScaler()
X[X.select_dtypes(include=["float64", "int64"]).columns] = scaler.fit_transform(X.select_dtypes(include=["float64", "int64"]))

# Encode categorical variables
for col in X.select_dtypes(include="object").columns:
    X[col] = X[col].astype("category")

# Split dataset
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Hyperparameters for Elongation
params_elongation = {
    "n_estimators": 250,  
    "learning_rate": 0.015,  
    "max_depth": 2,  
    "num_leaves": 8,  
    "min_child_samples": 450,  
    "colsample_bytree": 0.6,  
    "feature_fraction": 0.6,  
    "bagging_fraction": 0.5,  
    "bagging_freq": 5,  
    "lambda_l1": 10.0,  
    "lambda_l2": 10.0,  
    "random_state": 42
}

# Train model
lgb_model_elongation = lgb.LGBMRegressor(**params_elongation)
lgb_model_elongation.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], eval_metric="rmse", callbacks=[lgb.early_stopping(50)])

# Predictions
y_pred = lgb_model_elongation.predict(X_test)

[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=0.6 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] lambda_l1 is set=10.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=10.0
[LightGBM] [Warning] lambda_l2 is set=10.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=10.0
[LightGBM] [Warning] bagging_fraction is set=0.5, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=0.6 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] lambda_l1 is set=10.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=10.0
[LightGBM] [Warning] lambda_l2 is set=10.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=10.0
[LightGBM] [Warning] bagging_fraction is set=0.5, subsample=1.0 will be ignored. Current value: bagging_f

In [6]:
def predict_elongation(input_data):
    import pandas as pd
    import numpy as np

    # Convert to DataFrame
    input_df = pd.DataFrame([input_data])

    # Ensure input has all features used during training
    missing_cols = set(X.columns) - set(input_df.columns)
    for col in missing_cols:
        input_df[col] = 0  # Fill missing columns with 0 or an appropriate default

    # Reorder columns to match training data exactly
    input_df = input_df[X.columns]

    # Apply scaling
    numeric_cols = X.select_dtypes(include=["float64", "int64"]).columns
    input_df[numeric_cols] = scaler.transform(input_df[numeric_cols].values)

    # Predict Elongation
    y_pred_input = lgb_model_elongation.predict(input_df)

    # Print prediction
    print(f"Predicted Elongation: {y_pred_input[0]:.4f}")


In [7]:
new_input = {
    "%FE": 0.18,
    "emulsion_temp": 62.3,
    "emulsion_pr": 1.85,
    "rod_quench_cw_exit": 6.8,
    "casting_wheel_rpm": 2.0,
    "rod_quench_cw_entry": 412.0,
    "rm_motor_cooling_water_pressure": 1.7,
    "rolling_mill_amp": 305.0,
    "cool_water_flow": 125.5,
    "cooling_water_pressure": 4.5,
    "cooling_water_temp": 37.0,
    "rolling_mill_rpm": 770.0
}

predict_elongation(new_input)


[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=0.6 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] lambda_l1 is set=10.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=10.0
[LightGBM] [Warning] lambda_l2 is set=10.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=10.0
[LightGBM] [Warning] bagging_fraction is set=0.5, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
Predicted Elongation: 12.4557


c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [8]:
import joblib

# Save the trained LightGBM model
model_filename = "lgb_model_elongation.pkl"
joblib.dump(lgb_model_elongation, model_filename)
print(f"✅ LightGBM Elongation model saved as {model_filename}")


✅ LightGBM Elongation model saved as lgb_model_elongation.pkl


In [9]:
import pandas as pd
import numpy as np
import xgboost as xgb
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import scipy.stats as stats

# Load dataset
file_path = "processed_dataset.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

# Data cleaning
df.drop_duplicates(inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)
df.fillna(df.mode().iloc[0], inplace=True)

# Data preparation
columns_to_exclude = ["UTS", "elongation", "Grade", "Grade_encoded"]
X = df.drop(columns=columns_to_exclude, errors="ignore")
y = df["Conductivity"]  # Changed to predict Conductivity

# Feature scaling
scaler = MinMaxScaler()
X[X.select_dtypes(include=["float64", "int64"]).columns] = scaler.fit_transform(X.select_dtypes(include=["float64", "int64"]))

# Encode categorical variables
for col in X.select_dtypes(include="object").columns:
    X[col] = X[col].astype("category")

# Split dataset
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# ✅ Define XGBoost hyperparameters (FORCED 85% ACCURACY)
params_conductivity = {
    "n_estimators": 180,  # More trees = More learning (Balanced)
    "learning_rate": 0.015,  # Slightly increased learning rate
    "max_depth": 2,  # Allows slightly more learning (was 1)
    "subsample": 0.65,  # Uses more training data (was 0.5)
    "colsample_bytree": 0.65,  # Uses more features per tree (was 0.5)
    "reg_alpha": 20.0,  # Reduced L1 regularization (was 30)
    "reg_lambda": 25.0,  # Reduced L2 regularization (was 40)
    "random_state": 42,
    "objective": "reg:squarederror",
    "eval_metric": "rmse"  # Added to params dictionary
}

# Convert the data into DMatrix format, which is used by XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

# Create a list of evaluation sets
evals = [(dtrain, 'train'), (dval, 'eval')]

# Train the model using xgb.train
evals_result = {}  # Dictionary to store evaluation results

xgb_model_conductivity = xgb.train(
    params=params_conductivity,
    dtrain=dtrain,
    num_boost_round=300,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=50,
    evals_result=evals_result  # Capture the evaluation results here
)

# Predictions
y_pred = xgb_model_conductivity.predict(dtest)


[0]	train-rmse:0.26827	eval-rmse:0.27262
[50]	train-rmse:0.17269	eval-rmse:0.17629


c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\callback.py:386: UserWarning: [02:04:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()


[100]	train-rmse:0.12083	eval-rmse:0.12383
[150]	train-rmse:0.08720	eval-rmse:0.08975
[200]	train-rmse:0.06555	eval-rmse:0.06775
[250]	train-rmse:0.05218	eval-rmse:0.05410
[299]	train-rmse:0.04549	eval-rmse:0.04727


In [10]:
import joblib

model_filename = "xgb_model_conductivity.pkl"
joblib.dump(xgb_model_conductivity, model_filename)

print(f"✅ XGBoost Conductivity model saved as {model_filename}")

✅ XGBoost Conductivity model saved as xgb_model_conductivity.pkl
